# PCam metastasis detection - full run on Colab
**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.
Run the cells in order, one at a time. Each cell is safe to re-run.

## Cell 1: setup (GPU check, get the code)

In [ ]:
import torch, os
print("GPU available:", torch.cuda.is_available())
REPO_URL = "https://github.com/yuvxnv/pcam-metastasis-detection.git"
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!pip install -q h5py scikit-learn pandas matplotlib tqdm gdown

## Cell 2: data (about 7.5 GB; takes roughly 10 minutes)
It prints progress every 30 seconds. The long silent-looking stretch is the unpacking. Wait for `Data ready.`

In [ ]:
import os, subprocess, sys, time
ROOT, D = "/content/data", "/content/data/pcam"
NEED = [f"{D}/camelyonpatch_level_2_split_{s}_{k}.h5" for s in ("train", "valid", "test") for k in "xy"]
def have(): return all(os.path.exists(p) and os.path.getsize(p) > 10_000 for p in NEED)
def running(): return subprocess.run(["pgrep", "-f", "[d]ownload_data"], capture_output=True).returncode == 0

if have():
    print("Data already present. Nothing to do.")
else:
    if not running():
        os.makedirs(D, exist_ok=True)
        for f in os.listdir(D):  # clear leftovers from failed attempts
            p = f"{D}/{f}"
            if f.endswith(".part") or (f.endswith(".h5") and os.path.getsize(p) == 0):
                os.remove(p)
        log = open("/content/dl.log", "w")
        subprocess.Popen([sys.executable, "scripts/download_data.py", "--root", ROOT],
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print("Started download in the background.")
    else:
        print("A download is already running; waiting for it.")
    t0 = time.time()
    while running():  # safe to interrupt: the download keeps going; re-run this cell to resume watching
        sizes = {f[-40:]: round(os.path.getsize(f"{D}/{f}") / 1e9, 2) for f in sorted(os.listdir(D))}
        print(f"{int(time.time()-t0)//60:>3} min | GB on disk:", sizes, flush=True)
        time.sleep(30)
    print(open("/content/dl.log").read().replace("\r", "\n").split("\n")[-6:])
    assert have(), "Data still incomplete. Paste the last lines of /content/dl.log to Claude."
    for f in os.listdir(D):
        if f.endswith(".gz"): os.remove(f"{D}/{f}")
    print("Data ready.")

## Cell 3: look at the data (class balance + sample images)

In [ ]:
!python scripts/explore_data.py --root /content/data

## Cell 4: train both models (the long step)
Both models are trained on the full training set. The best epoch is picked using the **validation** set only.
Keep this tab open.

In [ ]:
EPOCHS = 12
!python -m src.train --model cnn --root /content/data --epochs $EPOCHS
!python -m src.train --model resnet18 --root /content/data --epochs $EPOCHS

## Cell 5: validation results (safe to repeat)
**Stop after this cell and show the output to Claude.** Do not run the next cell yet.

In [ ]:
!python -m src.evaluate --model cnn --root /content/data --split val
!python -m src.evaluate --model resnet18 --root /content/data --split val
import pandas as pd
display(pd.read_csv("results/history_cnn.csv"))
display(pd.read_csv("results/history_resnet18.csv"))

## Cell 6: FINAL test-set scoring. Run ONCE, only when told to.
A lock file blocks a second run.

In [ ]:
!python -m src.evaluate --model cnn --root /content/data --split test --final-test
!python -m src.evaluate --model resnet18 --root /content/data --split test --final-test

## Cell 7: download the results to your computer

In [ ]:
!zip -rq outputs.zip results checkpoints
from google.colab import files
files.download("outputs.zip")